# Day 1 — Notebook 04 (LangChain Edition)
# Embeddings, Similarity, Ranking & Reranking — Made Visible

**What changed from the brute-force version:** the underlying concepts (embeddings, cosine similarity,
ranking, filtering, reranking, top-k) are unchanged — but every mechanic is now built with the same
libraries you'd actually reach for in a production RAG stack:

| Brute-force notebook | This notebook |
|---|---|
| Raw `openai` client + manual JSON loading | `langchain-openai` chat/embedding models |
| Raw `chromadb` client + manual upsert | `langchain-chroma` vector store |
| Custom fixed/sentence/recursive/section splitters | `langchain-text-splitters` + **`SemanticChunker`** (new) |
| Manual `1 - distance` similarity math | `similarity_search_with_relevance_scores` |
| Manual metadata `where=` dict | Retriever `search_kwargs={"filter": ...}` |
| Vector search only | **Hybrid search** — `EnsembleRetriever` (BM25 + vector) |
| Custom LLM JSON-parsing reranker | `ContextualCompressionRetriever` + a real cross-encoder reranker |

### New in this version
- **Semantic chunking** — splitting text where meaning shifts, not just where character counts run out.
- **Hybrid retrieval** — combining keyword (BM25) and semantic (vector) search, which is what most real
  production RAG systems actually run in front of users.
- **A drop-in reranker** using LangChain's compression-retriever pattern, so swapping in a hosted reranker
  (Cohere, Voyage, etc.) later is a one-line change, not a rewrite.

### Core visual outputs
- Embedding summary.
- Pairwise semantic-similarity table.
- Ranked retrieval table with **relevance score**.
- Before/after metadata-filter comparison.
- Before/after **hybrid retrieval**.
- Before/after **reranking with rank movement**.


## 0. Install the LangChain stack

Run this once in your environment (network access required). Everything below assumes these packages
are already available — the same way the brute-force notebook assumed `openai` and `chromadb` were.


In [11]:
!pip install -U langchain langchain-openai langchain-chroma langchain-experimental \
     langchain-community langchain-text-splitters rank_bm25 sentence-transformers


  Using cached langchain-1.3.15-py3-none-any.whl.metadata (6.1 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached sentence_transformers-6.0.0-py3-none-any.whl.metadata (20 kB)
  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached transformers-5.15.0-py3-none-any.whl.metadata (32 kB)
  Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
Using cached langchain-1.3.15-py3-none-any.whl (147 kB)
Using cached langgraph-1.2.11-py3-none-any.whl (248 kB)
Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl (56 kB)
Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl (41 kB)
Using cached langgraph_sdk-0.4.2-

## 1. Azure setup — LangChain chat + embedding models

Instead of a raw `openai.OpenAI` client, we wrap the same Azure deployments in LangChain's model classes.
Everything downstream (vector store, retrievers, reranker, chains) can now be swapped for a different
provider (OpenAI, Bedrock, Ollama, ...) just by changing this one cell.


In [12]:
from pathlib import Path
import os, json, re
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from IPython.display import display

from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

ROOT = Path(".")
POLICY_DIR = ROOT / "data" / "healthcare_policies"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

load_dotenv(".env", override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
chat_deployment = os.getenv("AZURE_OPENAI_MODEL")
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

if not all([endpoint, api_key, chat_deployment, embedding_deployment]):
    raise ValueError(
        "Missing Azure configuration. Check AZURE_OPENAI_ENDPOINT, "
        "AZURE_OPENAI_API_KEY, AZURE_OPENAI_MODEL and "
        "AZURE_OPENAI_EMBEDDING_MODEL in .env"
    )

llm = AzureChatOpenAI(
    azure_endpoint=endpoint,
    api_key=api_key,
    api_version=api_version,
    azure_deployment=chat_deployment,
    temperature=0,
)

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=endpoint,
    api_key=api_key,
    api_version=api_version,
    azure_deployment=embedding_deployment,
)

env_df = pd.DataFrame([
    {"Component": "Chat / Generation", "Azure deployment": chat_deployment, "LangChain class": "AzureChatOpenAI"},
    {"Component": "Embeddings", "Azure deployment": embedding_deployment, "LangChain class": "AzureOpenAIEmbeddings"},
])
display(env_df)


,Component,Azure deployment,LangChain class
0,Chat / Generation,gpt-4.1-mini,AzureChatOpenAI
1,Embeddings,text-embedding-3-small,AzureOpenAIEmbeddings


## 2. Load the pre-chunked corpus (fixed / sentence / recursive / section)

Same source `.jsonl` artifacts as before. We convert each record into a LangChain `Document`
(`page_content` + `metadata`) — the common currency every LangChain retriever, splitter and
vector store speaks.


In [13]:
from langchain_core.documents import Document

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def records_to_documents(records):
    docs = []
    for r in records:
        meta = {k: v for k, v in r.items() if k != "text"}
        docs.append(Document(page_content=r["text"], metadata=meta))
    return docs

records_by_strategy = {
    s: load_jsonl(ARTIFACT_DIR / f"policy_chunks_{s}.jsonl")
    for s in ["fixed", "sentence", "recursive", "section"]
}

documents_by_strategy = {
    s: records_to_documents(records) for s, records in records_by_strategy.items()
}

corpus_df = pd.DataFrame([
    {
        "Strategy": s,
        "Chunks": len(docs),
        "Avg chars": round(np.mean([len(d.page_content) for d in docs]), 1),
    }
    for s, docs in documents_by_strategy.items()
])
display(corpus_df)


,Strategy,Chunks,Avg chars
0,fixed,16,581.8
1,sentence,17,482.2
2,recursive,17,539.5
3,section,34,214.7


## 3. NEW — Semantic chunking with `SemanticChunker`

Fixed-size and recursive splitters cut text at character/token boundaries and hope a sentence or
section boundary lines up nearby. **Semantic chunking** does something different: it embeds
consecutive sentences, measures how similar each sentence is to the one before it, and cuts a new
chunk exactly where that similarity **drops** — i.e. where the topic changes. The result is chunks
that track meaning rather than length.

Because our `.jsonl` artifacts are already pre-chunked (by an earlier notebook), we first
reconstruct each document's raw running text by re-joining its `section` chunks in order. In a real
pipeline you would run `SemanticChunker` directly on the original raw document text, before any other
splitting — this reconstruction step is just so this notebook is self-contained.


In [15]:
from langchain_experimental.text_splitter import SemanticChunker

def reconstruct_raw_documents(section_records):
    # Rebuild approximate raw per-document text from already-chunked 'section' records.
    by_doc = {}
    for r in section_records:
        by_doc.setdefault(r["doc_id"], {"meta": r, "parts": []})
        by_doc[r["doc_id"]]["parts"].append(r["text"])
    raw_docs = {}
    for doc_id, payload in by_doc.items():
        raw_docs[doc_id] = {
            "text": "\n\n".join(payload["parts"]),
            "meta": payload["meta"],
        }
    return raw_docs

raw_docs_by_id = reconstruct_raw_documents(records_by_strategy["section"])

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",   # also: "standard_deviation", "interquartile", "gradient"
    breakpoint_threshold_amount=90,            # higher = fewer, larger chunks
)

semantic_documents = []
for doc_id, payload in raw_docs_by_id.items():
    base_meta = {k: v for k, v in payload["meta"].items() if k != "text"}
    base_meta = {**base_meta, "chunk_strategy": "semantic"}
    chunks = semantic_splitter.create_documents(
        texts=[payload["text"]],
        metadatas=[base_meta],
    )
    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = f"{doc_id}-semantic-{i}"
    semantic_documents.extend(chunks)

documents_by_strategy["semantic"] = semantic_documents

corpus_df = pd.DataFrame([
    {
        "Strategy": s,
        "Chunks": len(docs),
        "Avg chars": round(np.mean([len(d.page_content) for d in docs]), 1),
    }
    for s, docs in documents_by_strategy.items()
])
display(corpus_df)


NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}

> **Teaching point:** compare the `semantic` row above with `fixed`, `sentence`, `recursive` and
> `section`. Semantic chunking usually produces a *variable* chunk count and size — because it's
> reacting to where the content actually changes topic, not to a fixed rule. Try changing
> `breakpoint_threshold_amount` and re-running to see the trade-off between larger, more complete
> chunks and smaller, more precise ones.


## 4. What does an embedding actually look like?

Same idea as the brute-force notebook, just using the LangChain embeddings interface
(`embed_documents` / `embed_query`) instead of calling the OpenAI client directly.


In [ ]:
sample_texts = [
    "MRI scans require prior authorization.",
    "Advanced imaging needs insurer approval.",
    "The member updated a mailing address.",
]

sample_vectors = embeddings.embed_documents(sample_texts)

embedding_df = pd.DataFrame([
    {
        "Text": text,
        "Vector Dimensions": len(vec),
        "First 6 Values": str([round(v, 4) for v in vec[:6]]),
        "Vector Norm": round(float(np.linalg.norm(vec)), 4),
    }
    for text, vec in zip(sample_texts, sample_vectors)
])
display(embedding_df)

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

pairs = [
    (0, 1, "Same business meaning, different wording"),
    (0, 2, "Different business meaning"),
    (1, 2, "Different business meaning"),
]

sim_rows = []
for i, j, relationship in pairs:
    sim_rows.append({
        "Text A": sample_texts[i],
        "Text B": sample_texts[j],
        "Expected Relationship": relationship,
        "Cosine Similarity": round(cosine_similarity(sample_vectors[i], sample_vectors[j]), 4),
    })

display(pd.DataFrame(sim_rows))


NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}

> Embeddings let "insurer approval" retrieve "prior authorization" even though the exact terms differ.


## 5. Persist every strategy as a Chroma collection — the LangChain way

`langchain-chroma`'s `Chroma.from_documents(...)` handles embedding + upserting in a single call —
no manual batching, id management or metadata dict-building required. Each chunking strategy
(including the new `semantic` one) gets its own persistent collection so we can compare them later.


In [ ]:
from langchain_chroma import Chroma

VECTOR_DB_PATH = str(ARTIFACT_DIR / "chroma_policy_db_langchain")

vectorstores = {}
for strategy, docs in documents_by_strategy.items():
    vectorstores[strategy] = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        collection_name=f"policy_{strategy}",
        persist_directory=VECTOR_DB_PATH,
        collection_metadata={"hnsw:space": "cosine"},
    )

vector_store_df = pd.DataFrame([
    {"Collection": f"policy_{s}", "Strategy": s, "Vectors Stored": vs._collection.count()}
    for s, vs in vectorstores.items()
])
display(vector_store_df)


## 6. Ranked semantic retrieval — no more manual distance math

`similarity_search_with_relevance_scores` returns an already-normalized 0–1 relevance score, so we
no longer need the `1 - distance` conversion the brute-force notebook did by hand.


In [ ]:
def search_to_df(results):
    rows = []
    for rank, (doc, score) in enumerate(results, start=1):
        meta = doc.metadata
        rows.append({
            "Rank": rank,
            "Relevance Score ↑": round(score, 4),
            "Document": meta.get("doc_id"),
            "Plan": meta.get("plan_type"),
            "Section": meta.get("section"),
            "Page": meta.get("page"),
            "Chunk Strategy": meta.get("chunk_strategy"),
            "Retrieved Text": re.sub(r"\s+", " ", doc.page_content)[:240] + "...",
        })
    return pd.DataFrame(rows)

question = "For Gold PPO, when does physical therapy start requiring authorization?"
results = vectorstores["section"].similarity_search_with_relevance_scores(question, k=5)
display(search_to_df(results))


### Reading this ranking

**Rank 1** is the closest semantic match. The relevance score is comparable **within the same
embedding model and use case**, but it's not a universal confidence percentage.

> **Important:** A high relevance score means "semantically close," not "factually correct."


## 7. Metadata filtering — retrievers, not raw `where=` dicts

Wrapping the vector store as a retriever (`as_retriever`) is what lets it plug into every other
LangChain component (compression, ensembles, chains) later. Metadata filtering becomes a
`search_kwargs["filter"]` argument on that retriever.


In [ ]:
query = "When does physical therapy require authorization?"

retriever_unfiltered = vectorstores["section"].as_retriever(search_kwargs={"k": 5})
retriever_filtered = vectorstores["section"].as_retriever(
    search_kwargs={"k": 5, "filter": {"plan_type": "Silver HMO"}}
)

def retriever_to_df(retriever, label):
    docs = retriever.invoke(query)
    rows = []
    for rank, doc in enumerate(docs, start=1):
        meta = doc.metadata
        rows.append({
            "Search Mode": label,
            "Rank": rank,
            "Document": meta.get("doc_id"),
            "Plan": meta.get("plan_type"),
            "Section": meta.get("section"),
            "Retrieved Text": re.sub(r"\s+", " ", doc.page_content)[:200] + "...",
        })
    return pd.DataFrame(rows)

before = retriever_to_df(retriever_unfiltered, "Semantic only")
after = retriever_to_df(retriever_filtered, "Semantic + Silver HMO filter")

display(pd.concat([before, after], ignore_index=True))


> **Takeaway:** Filtering changes the candidate universe *before* ranking. It stops the retriever
> from surfacing the wrong plan even when its text is highly similar.


## 8. Compare retrieval across five chunking strategies (including semantic)


In [ ]:
query = "What information is needed for continued physical therapy authorization?"

strategy_results = []
for strategy, vs in vectorstores.items():
    results = vs.similarity_search_with_relevance_scores(query, k=3)
    for rank, (doc, score) in enumerate(results, start=1):
        strategy_results.append({
            "Strategy": strategy,
            "Rank": rank,
            "Relevance": round(score, 4),
            "Document": doc.metadata.get("doc_id"),
            "Section": doc.metadata.get("section"),
            "Text": re.sub(r"\s+", " ", doc.page_content)[:200] + "...",
        })

display(pd.DataFrame(strategy_results))


> **Teaching question:** "Which chunking method — including the new semantic strategy — puts the
> complete business rule closest to the top?"


## 9. NEW — Hybrid retrieval: BM25 + vector search via `EnsembleRetriever`

Pure vector search misses exact keyword matches (plan codes, section numbers, precise legal
phrasing) that a classic keyword search nails instantly. Most production RAG systems run **both**
and blend the results. LangChain's `EnsembleRetriever` does this with reciprocal-rank fusion in a
few lines — no custom scoring logic required.


In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

section_docs = documents_by_strategy["section"]

bm25_retriever = BM25Retriever.from_documents(section_docs)
bm25_retriever.k = 5

vector_retriever = vectorstores["section"].as_retriever(search_kwargs={"k": 5})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],   # tilt slightly toward semantic, keep keyword precision
)

query = "Silver HMO SECTION 3 physical therapy prior authorization"

def docs_to_df(docs, label):
    rows = []
    for rank, doc in enumerate(docs, start=1):
        meta = doc.metadata
        rows.append({
            "Retriever": label,
            "Rank": rank,
            "Document": meta.get("doc_id"),
            "Section": meta.get("section"),
            "Text": re.sub(r"\s+", " ", doc.page_content)[:180] + "...",
        })
    return pd.DataFrame(rows)

vector_only_df = docs_to_df(vector_retriever.invoke(query), "Vector only")
hybrid_df = docs_to_df(hybrid_retriever.invoke(query), "Hybrid (BM25 + vector)")

display(pd.concat([vector_only_df, hybrid_df], ignore_index=True))


> **Takeaway:** Hybrid retrieval catches candidates that pure vector search can rank low —
> exact section numbers, plan codes and rare terms — while still benefiting from semantic recall.
> This is the retrieval pattern most teams reach for in production before ever touching a reranker.


## 10. Reranking — a real cross-encoder, not a hand-rolled JSON prompt

The brute-force notebook asked the chat model to score candidates and hand-parsed the JSON response.
That works, but it's slow, costs a chat-completion call per query, and is brittle to parse.

`ContextualCompressionRetriever` + `CrossEncoderReranker` gives you the same "rerank the top
candidates" behavior using a purpose-built cross-encoder model that scores (question, chunk) pairs
directly — no prompt engineering, no JSON parsing. Swapping this for a hosted reranker (Cohere,
Voyage, etc.) later is a one-line change: replace `HuggingFaceCrossEncoder` with the hosted
reranker's LangChain wrapper.


In [ ]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever

cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker = CrossEncoderReranker(model=cross_encoder, top_n=6)

base_retriever = vectorstores["section"].as_retriever(search_kwargs={"k": 6})
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever,
)

rerank_question = "What clinical documentation is required when requesting continued physical therapy?"

vector_order = base_retriever.invoke(rerank_question)
reranked_order = compression_retriever.invoke(rerank_question)

def rank_lookup(docs):
    return {id(d.metadata.get("chunk_id", d.page_content[:30])): i + 1 for i, d in enumerate(docs)}

def chunk_key(doc):
    return doc.metadata.get("chunk_id", doc.page_content[:30])

vector_rank = {chunk_key(d): i + 1 for i, d in enumerate(vector_order)}
rerank_rank = {chunk_key(d): i + 1 for i, d in enumerate(reranked_order)}

rows = []
for doc in reranked_order:
    key = chunk_key(doc)
    rows.append({
        "Vector Rank": vector_rank.get(key),
        "Rank After Rerank": rerank_rank.get(key),
        "Rank Movement": (vector_rank.get(key, 0) - rerank_rank.get(key, 0)),
        "Rerank Score ↑": round(doc.metadata.get("relevance_score", 0), 4),
        "Document": doc.metadata.get("doc_id"),
        "Section": doc.metadata.get("section"),
        "Text": re.sub(r"\s+", " ", doc.page_content)[:180] + "...",
    })

display(pd.DataFrame(rows))


### The reranking output

- **Vector Rank** = broad semantic closeness from the vector store alone.
- **Rerank Score** = the cross-encoder's direct (question, chunk) relevance score.
- **Rank Movement > 0** = chunk moved upward after reranking.
- **Rank Movement < 0** = chunk moved downward.

> **Takeaway:** Retrieval (and hybrid retrieval) finds candidates; reranking decides which
> candidates deserve the scarce top context positions — and doing it with a purpose-built
> cross-encoder is both cheaper and more consistent than asking a chat model to score JSON by hand.


## 11. Top-k — convert the trade-off into a visible summary

Same trade-off as before, now expressed through a retriever's `search_kwargs["k"]`.


In [ ]:
rows = []
query = "Gold PPO chiropractic annual visit limit"

for k in [1, 3, 5, 8]:
    retriever_k = vectorstores["section"].as_retriever(search_kwargs={"k": k})
    docs = retriever_k.invoke(query)
    scored = vectorstores["section"].similarity_search_with_relevance_scores(query, k=k)
    best_score = max(score for _, score in scored) if scored else None
    rows.append({
        "Top-K": k,
        "Best Relevance": round(best_score, 4) if best_score is not None else None,
        "Unique Documents": len({d.metadata.get("doc_id") for d in docs}),
        "Unique Plans": len({d.metadata.get("plan_type") for d in docs}),
        "Retrieved Characters": sum(len(d.page_content) for d in docs),
        "Top Result": f"{docs[0].metadata.get('doc_id')} | {docs[0].metadata.get('section')}" if docs else None,
    })

display(pd.DataFrame(rows))


> **Takeaway:** Top-k is a recall-versus-noise control. Higher k increases the evidence pool but
> also increases downstream context, token cost and distraction — same conclusion as the
> brute-force notebook, just reached through the retriever abstraction you'll actually use in
> production.


## Day 1 completion (LangChain edition)

You can now explain and *build* the complete retrieval mechanics with production-grade libraries:

**Chunk strategy (fixed / sentence / recursive / section / semantic) → LangChain embeddings →
Chroma vector store → ranked retrieval → metadata filtering → hybrid (BM25 + vector) retrieval →
cross-encoder reranking → top-k**

Day 2 will show how these ranked chunks become grounded answers using an LCEL RAG chain — and what
happens when the evidence is missing.
